# 03 — 성능 평가 및 모델 카드 작성

**목적:** 베이스 모델(zero-shot) vs 파인튜닝 모델을 `test.jsonl` 100개로 비교.  
결과 테이블을 HuggingFace 모델 카드에 그대로 붙여넣을 수 있는 형태로 출력.

**런타임:** T4 GPU 필수  
**측정 지표:**
- `skill_f1` — 추출한 기술명 집합의 F1 (정규화 후 비교)
- `category_acc` — 맞게 추출된 기술의 category 정답률
- `importance_acc` — required/preferred 분류 정확도
- `json_parse_rate` — 출력이 valid JSON인 비율 (포맷 준수)

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q unsloth transformers bitsandbytes huggingface_hub

In [ ]:
import json
import os
from pathlib import Path

import torch

if IN_COLAB:
    from google.colab import userdata, drive
    drive.mount('/content/drive')
    HF_TOKEN   = userdata.get('HF_TOKEN', '')
    HF_REPO_ID = userdata.get('HF_REPO_ID', '')  # 파인튜닝된 모델 repo
    DATASET_DIR   = Path('/content/drive/MyDrive/job-skill-ft/dataset')
    ADAPTER_PATH  = Path('/content/drive/MyDrive/job-skill-ft/checkpoints/best_adapter')
else:
    from dotenv import load_dotenv
    load_dotenv()
    HF_TOKEN   = os.getenv('HF_TOKEN', '')
    HF_REPO_ID = os.getenv('HF_REPO_ID', '')
    DATASET_DIR  = Path('finetune/dataset')
    ADAPTER_PATH = Path('finetune/checkpoints/best_adapter')

BASE_MODEL_NAME = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'
MAX_SEQ_LEN = 1024

print(f'CUDA: {torch.cuda.is_available()}')
print(f'테스트 데이터: {DATASET_DIR / "test.jsonl"}')
print(f'파인튜닝 어댑터: {ADAPTER_PATH if ADAPTER_PATH.exists() else HF_REPO_ID or "(없음)"}')

In [ ]:
# ── 테스트 데이터 로드 ─────────────────────────────────────────
def load_jsonl(path: Path) -> list[dict]:
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

test_samples = load_jsonl(DATASET_DIR / 'test.jsonl')
print(f'테스트 샘플: {len(test_samples)}개')

# ground truth 파싱
test_data = []
for s in test_samples:
    try:
        gt = json.loads(s['output'])
        test_data.append({
            'instruction': s['instruction'],
            'input': s['input'],
            'ground_truth': gt,
        })
    except Exception:
        pass

print(f'유효 테스트 샘플: {len(test_data)}개')

In [ ]:
# ── 평가 지표 함수 ─────────────────────────────────────────────
import re


def normalize_skill_name(name: str) -> str:
    """비교를 위해 기술명 정규화 (소문자, 특수문자 제거)."""
    name = name.lower().strip()
    name = re.sub(r'[.\-_/]', '', name)  # react.js → reactjs
    name = re.sub(r'\s+', '', name)       # hugging face → huggingface
    return name


def parse_output(raw: str) -> dict | None:
    """모델 출력 → JSON 파싱. 실패 시 None."""
    try:
        clean = raw.strip()
        # 마크다운 펜스 제거
        clean = re.sub(r'^```json\s*', '', clean)
        clean = re.sub(r'```\s*$', '', clean)
        clean = clean.strip()
        return json.loads(clean)
    except Exception:
        return None


def compute_metrics(predictions: list[dict | None], ground_truths: list[dict]) -> dict:
    """예측 결과 목록과 정답 목록으로 지표를 계산."""
    assert len(predictions) == len(ground_truths)

    n = len(predictions)
    parse_successes = 0
    f1_sum = 0.0
    category_correct = 0
    category_total = 0
    importance_correct = 0
    importance_total = 0

    for pred, truth in zip(predictions, ground_truths):
        if pred is None:
            continue
        parse_successes += 1

        pred_skills  = pred.get('skills', [])
        truth_skills = truth.get('skills', [])

        # Skill F1 — 정규화된 이름 기준 set 비교
        pred_names  = {normalize_skill_name(s.get('raw', '')) for s in pred_skills}
        truth_names = {normalize_skill_name(s.get('raw', '')) for s in truth_skills}

        tp = len(pred_names & truth_names)
        fp = len(pred_names - truth_names)
        fn = len(truth_names - pred_names)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        f1_sum += f1

        # Category / Importance 정확도 — 이름이 일치하는 기술만 비교
        truth_map = {
            normalize_skill_name(s.get('raw', '')): s
            for s in truth_skills
        }
        for ps in pred_skills:
            norm_name = normalize_skill_name(ps.get('raw', ''))
            if norm_name in truth_map:
                ts = truth_map[norm_name]
                category_total += 1
                if ps.get('category') == ts.get('category'):
                    category_correct += 1
                importance_total += 1
                if ps.get('importance') == ts.get('importance'):
                    importance_correct += 1

    return {
        'json_parse_rate' : parse_successes / n,
        'skill_f1'        : f1_sum / n,
        'category_acc'    : category_correct / category_total if category_total > 0 else 0.0,
        'importance_acc'  : importance_correct / importance_total if importance_total > 0 else 0.0,
        'n_samples'       : n,
    }


print('평가 함수 정의 완료')

In [ ]:
# ── 추론 헬퍼 ─────────────────────────────────────────────────
from unsloth import FastLanguageModel

SYSTEM_MSG = (
    'You are a technical skill extractor. '
    'Extract skills from job postings and return valid JSON only. '
    'No explanation or markdown fences.'
)


def run_inference(model, tokenizer, test_items: list[dict], batch_size: int = 8) -> list[dict | None]:
    """테스트 샘플 전체에 대해 추론 실행. 파싱된 dict 또는 None 반환."""
    FastLanguageModel.for_inference(model)
    results = []

    for i in range(0, len(test_items), batch_size):
        batch = test_items[i:i + batch_size]
        prompts = []
        for item in batch:
            messages = [
                {'role': 'system',   'content': SYSTEM_MSG},
                {'role': 'user',     'content': f"{item['instruction']}\n\n{item['input']}"},
            ]
            prompts.append(
                tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            )

        enc = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True,
                        max_length=MAX_SEQ_LEN).to('cuda')
        with torch.no_grad():
            gen = model.generate(
                **enc,
                max_new_tokens=512,
                temperature=0.0,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        for j, ids in enumerate(gen):
            new_ids = ids[enc.input_ids.shape[1]:]
            text = tokenizer.decode(new_ids, skip_special_tokens=True)
            results.append(parse_output(text))

        if (i // batch_size + 1) % 5 == 0:
            print(f'  [{i + len(batch)}/{len(test_items)}] 처리 완료')

    return results


print('추론 헬퍼 정의 완료')

In [ ]:
# ── [1/2] 베이스 모델 평가 (zero-shot) ────────────────────────
print('베이스 모델 로드 중...')
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

print('베이스 모델 추론 중...')
base_preds = run_inference(base_model, base_tokenizer, test_data)
base_metrics = compute_metrics(base_preds, [d['ground_truth'] for d in test_data])
print('\n베이스 모델 결과:')
for k, v in base_metrics.items():
    print(f'  {k:20} : {v:.4f}' if isinstance(v, float) else f'  {k:20} : {v}')

# 메모리 해제
del base_model
torch.cuda.empty_cache()

In [ ]:
# ── [2/2] 파인튜닝 모델 평가 ──────────────────────────────────
ft_source = str(ADAPTER_PATH) if ADAPTER_PATH.exists() else HF_REPO_ID

if not ft_source:
    print('[skip] 파인튜닝 모델 경로가 없습니다. 02_finetune.ipynb를 먼저 실행하거나 HF_REPO_ID를 설정하세요.')
    ft_metrics = None
else:
    print(f'파인튜닝 모델 로드 중: {ft_source}')
    ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
        model_name=ft_source,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=True,
        token=HF_TOKEN if HF_TOKEN else None,
    )

    print('파인튜닝 모델 추론 중...')
    ft_preds = run_inference(ft_model, ft_tokenizer, test_data)
    ft_metrics = compute_metrics(ft_preds, [d['ground_truth'] for d in test_data])
    print('\n파인튜닝 모델 결과:')
    for k, v in ft_metrics.items():
        print(f'  {k:20} : {v:.4f}' if isinstance(v, float) else f'  {k:20} : {v}')

    del ft_model
    torch.cuda.empty_cache()

In [ ]:
# ── 비교 테이블 출력 ────────────────────────────────────────────
# Claude Haiku는 ground truth 생성자이므로 정의상 F1=1.0
haiku_metrics = {
    'json_parse_rate': 1.0,
    'skill_f1': 1.0,
    'category_acc': 1.0,
    'importance_acc': 1.0,
}

rows = [
    ('Claude Haiku (레이블 생성자)',        haiku_metrics),
    ('Qwen2.5-1.5B base (zero-shot)',     base_metrics),
]
if ft_metrics:
    rows.append(('Qwen2.5-1.5B **파인튜닝**', ft_metrics))

METRICS = ['json_parse_rate', 'skill_f1', 'category_acc', 'importance_acc']
LABELS  = ['JSON Parse Rate', 'Skill F1', 'Category Acc', 'Importance Acc']

# 터미널 출력
header = f'{"모델":42}' + ''.join(f'  {l:14}' for l in LABELS)
print('\n=== 최종 성능 비교 ===')
print(header)
print('-' * len(header))
for name, m in rows:
    row = f'{name:42}' + ''.join(f'  {m[k]:.4f}{"":8}' for k in METRICS)
    print(row)

# 마크다운 테이블 (모델 카드용)
md_table = '| 모델 | ' + ' | '.join(LABELS) + ' |\n'
md_table += '|' + '|'.join(['---'] * (len(LABELS) + 1)) + '|\n'
for name, m in rows:
    md_table += f'| {name} | ' + ' | '.join(f'{m[k]:.3f}' for k in METRICS) + ' |\n'

print('\n=== 모델 카드용 마크다운 테이블 ===')
print(md_table)

In [ ]:
# ── 모델 카드 전체 초안 생성 ────────────────────────────────────
REPO_ID = HF_REPO_ID or 'your_username/qwen2.5-1.5b-job-skill-extractor'

ft_f1 = ft_metrics['skill_f1'] if ft_metrics else 0.0
ft_parse = ft_metrics['json_parse_rate'] if ft_metrics else 0.0

model_card = f'''---
language: en
license: apache-2.0
pipeline_tag: text-generation
tags:
  - skill-extraction
  - job-posting
  - information-extraction
  - qlora
  - unsloth
  - qwen2.5
base_model: Qwen/Qwen2.5-1.5B-Instruct
datasets:
  - custom (Adzuna job postings, labeled by Claude Haiku)
metrics:
  - f1
---

# qwen2.5-1.5b-job-skill-extractor

Qwen2.5-1.5B-Instruct fine-tuned with QLoRA to extract technical skills from English job postings.
Part of the **Job Skill Analyzer** project — an Agentic RAG system for career gap analysis.

## 학습 데이터

- Adzuna API 수집 영어 채용공고 (AI/ML/Backend 직군 위주)
- Claude Haiku (`claude-haiku-4-5-20251001`) 자동 레이블링
- Train {len(train_samples)}개 / Test {len(test_data)}개

## 성능

Test set {len(test_data)}개 기준 (ground truth = Claude Haiku 레이블):

{md_table}

## 사용 예시

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch, json

base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct", torch_dtype=torch.float16, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("{REPO_ID}")
model = PeftModel.from_pretrained(base, "{REPO_ID}")

messages = [
    {{"role": "system",   "content": "You are a technical skill extractor. Return valid JSON only."}},
    {{"role": "user",     "content": "Title: Senior AI Engineer\\n\\nDescription: Required Python, LangGraph..."}},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.0, do_sample=False)
response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
result = json.loads(response)
```

## 출력 형식

```json
{{
  "job_title": "Senior AI Engineer",
  "skills": [
    {{"raw": "Python",    "category": "language",  "importance": "required"}},
    {{"raw": "LangGraph", "category": "framework", "importance": "required"}},
    {{"raw": "Neo4j",     "category": "database",  "importance": "preferred"}}
  ]
}}
```

## 한계

- 영어 공고 기반 학습 → 한국어 공고 성능 미검증
- AI/ML/Backend 직군 중심 → 의료·법률 등 타 도메인 기술 추출 precision 낮을 수 있음
- Ground truth가 Claude Haiku이므로 Haiku의 오류 패턴을 학습했을 가능성 있음

## 관련 프로젝트

[Job Skill Analyzer](https://github.com/K-ismyname/pj1) — 채용공고 수집·분석 + 이력서 갭 분석 Agentic RAG 시스템
'''

print(model_card)

# 파일로 저장
card_path = Path('finetune/MODEL_CARD.md') if not IN_COLAB else Path('/content/drive/MyDrive/job-skill-ft/MODEL_CARD.md')
card_path.parent.mkdir(parents=True, exist_ok=True)
card_path.write_text(model_card, encoding='utf-8')
print(f'\n모델 카드 저장: {card_path}')

In [ ]:
# ── (선택) HF Hub에 모델 카드 업로드 ─────────────────────────
if HF_TOKEN and HF_REPO_ID:
    from huggingface_hub import HfApi

    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj=model_card.encode('utf-8'),
        path_in_repo='README.md',
        repo_id=HF_REPO_ID,
        repo_type='model',
    )
    print(f'모델 카드 업로드 완료: https://huggingface.co/{HF_REPO_ID}')
else:
    print('[skip] HF_TOKEN 또는 HF_REPO_ID가 없어 업로드 생략.')
    print('  MODEL_CARD.md를 HF Hub 리포지토리의 README.md로 수동 업로드하거나,')
    print('  Secrets에 HF_TOKEN, HF_REPO_ID를 설정 후 이 셀을 재실행하세요.')